In [ ]:
"""
ZAI — a conversational knowledge-base search assistant
================================================================

Features
--------
1. A searchable knowledge base (plain in-memory list — no database
   engine required) pre-seeded with facts across Science, Math,
   English/Grammar, and General Trivia categories.
2. A ranked search engine (keyword overlap + fuzzy matching) so queries
   don't need to match text exactly.
3. A safe math expression evaluator (handles +, -, *, /, ^, %, sqrt,
   sin/cos/tan, factorial, etc.) without using Python's dangerous eval().
4. A lightweight conversational layer for greetings, small talk, and
   general chit-chat, using pattern matching.
5. An intent router that decides whether an input is: a math problem,
   a knowledge/database query, small talk, or a request to add new
   knowledge to the database.
6. A simple command-line chat loop to interact with everything above.

Run it with:  python advanced_ai.py

No third-party dependencies are required — everything uses the
Python standard library.
"""

from __future__ import annotations

import ast
import difflib
import math
import operator
import os
import random
import re
from datetime import datetime
from typing import List, Optional, Tuple


# ============================================================
# 1. KNOWLEDGE DATABASE
# ============================================================

class KnowledgeDatabase:
    """
    A small knowledge base with ranked search, backed by a plain Python
    list of dictionaries (no database engine required — this runs
    anywhere Python does, including restricted online compilers that
    block modules like sqlite3).
    """

    def __init__(self):
        self.entries: List[dict] = []
        self._next_id = 1
        self._idf: dict = {}
        self._default_idf: float = 1.0
        self._seed_data()
        self._build_idf_index()

    # ---------- seeding ----------

    def _seed_data(self) -> None:
        raw_entries = [
            # --- Science ---
            ("science", "Photosynthesis",
             "Photosynthesis is the process plants use to convert sunlight, "
             "water, and carbon dioxide into glucose and oxygen.",
             "photosynthesis plants sunlight glucose oxygen carbon dioxide"),
            ("science", "Gravity",
             "Gravity is the force of attraction between two masses. On Earth "
             "it accelerates objects toward the ground at about 9.8 m/s^2.",
             "gravity force mass acceleration earth newton"),
            ("science", "Atom",
             "An atom is the basic unit of matter, made of protons, neutrons, "
             "and electrons. Protons and neutrons form the nucleus.",
             "atom proton neutron electron nucleus matter"),
            ("science", "States of Matter",
             "The common states of matter are solid, liquid, gas, and plasma, "
             "distinguished by particle arrangement and energy.",
             "states matter solid liquid gas plasma"),
            ("science", "DNA",
             "DNA (deoxyribonucleic acid) carries genetic instructions for the "
             "development and function of living organisms, structured as a "
             "double helix.",
             "dna genetics double helix genes chromosome"),
            ("science", "Speed of Light",
             "The speed of light in a vacuum is approximately 299,792 kilometers "
             "per second (about 186,282 miles per second).",
             "speed light vacuum km fast constant physics"),
            ("science", "Water Cycle",
             "The water cycle describes evaporation, condensation, precipitation, "
             "and collection of water moving through Earth's systems.",
             "water cycle evaporation condensation precipitation"),
            ("science", "Cell",
             "A cell is the smallest structural and functional unit of life. "
             "Plant cells have a cell wall and chloroplasts; animal cells do not.",
             "cell biology plant animal chloroplast wall organism"),
            ("science", "Periodic Table",
             "The periodic table arranges all known chemical elements by "
             "increasing atomic number, with columns (groups) sharing similar "
             "chemical properties.",
             "periodic table elements atomic number chemistry groups"),
            ("science", "Newton's Laws of Motion",
             "1) An object stays at rest or in motion unless acted on by a force. "
             "2) Force equals mass times acceleration (F=ma). 3) For every "
             "action there is an equal and opposite reaction.",
             "newton laws motion force mass acceleration inertia physics"),
            ("science", "Ecosystem",
             "An ecosystem is a community of living organisms interacting with "
             "each other and their physical environment, exchanging energy and "
             "nutrients.",
             "ecosystem biology environment organisms food chain"),
            ("science", "Solar System",
             "Our solar system has eight planets orbiting the Sun: Mercury, "
             "Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune.",
             "solar system planets sun mercury venus earth mars jupiter saturn uranus neptune"),
            ("science", "Chemical vs Physical Change",
             "A physical change alters form but not composition (e.g. melting "
             "ice). A chemical change produces a new substance with different "
             "properties (e.g. burning wood).",
             "chemical physical change reaction matter composition"),
            ("science", "Electricity and Circuits",
             "Electric current is the flow of electric charge through a "
             "conductor. A circuit needs a closed loop, a power source, and a "
             "load (like a bulb) to function.",
             "electricity circuit current voltage resistance ohm"),
            ("science", "Human Digestive System",
             "The digestive system breaks food into nutrients the body can "
             "absorb, moving it through the mouth, esophagus, stomach, and "
             "intestines.",
             "digestive system stomach intestine nutrients digestion"),
            ("science", "Sound Waves",
             "Sound travels as a mechanical wave caused by vibrations, needing "
             "a medium (like air or water) to move — it cannot travel through "
             "a vacuum.",
             "sound wave vibration frequency pitch medium"),

            # --- Math ---
            ("math", "Pythagorean Theorem",
             "In a right triangle, a^2 + b^2 = c^2, where c is the hypotenuse "
             "and a, b are the other two sides.",
             "pythagorean theorem triangle hypotenuse right angle"),
            ("math", "Prime Numbers",
             "A prime number is a whole number greater than 1 that has no "
             "positive divisors other than 1 and itself, e.g. 2, 3, 5, 7, 11.",
             "prime number divisor factor"),
            ("math", "Order of Operations",
             "PEMDAS: Parentheses, Exponents, Multiplication and Division "
             "(left to right), then Addition and Subtraction (left to right).",
             "order operations pemdas bodmas parentheses exponents"),
            ("math", "Pi",
             "Pi (π) is the ratio of a circle's circumference to its diameter, "
             "approximately 3.14159, and is an irrational number.",
             "pi circle circumference diameter irrational constant"),
            ("math", "Quadratic Formula",
             "For ax^2 + bx + c = 0, the solutions are x = (-b ± sqrt(b^2 - 4ac)) / (2a).",
             "quadratic formula equation roots solve"),
            ("math", "Fibonacci Sequence",
             "The Fibonacci sequence starts 0, 1, 1, 2, 3, 5, 8, 13... where each "
             "number is the sum of the two preceding ones.",
             "fibonacci sequence golden ratio numbers"),
            ("math", "Fractions",
             "A fraction represents part of a whole, written as numerator over "
             "denominator (e.g. 3/4). To add fractions, find a common "
             "denominator first.",
             "fraction numerator denominator part whole add subtract"),
            ("math", "Percentages",
             "A percentage expresses a number as a part of 100. To find X% of "
             "a number, multiply the number by X and divide by 100.",
             "percentage percent ratio proportion"),
            ("math", "Area and Perimeter",
             "Perimeter is the distance around a shape's edges; area is the "
             "space inside it. A rectangle's area is length times width; its "
             "perimeter is 2 times (length + width).",
             "area perimeter rectangle square shape geometry"),
            ("math", "Mean, Median, and Mode",
             "Mean is the average (sum divided by count). Median is the middle "
             "value when sorted. Mode is the most frequently occurring value.",
             "mean median mode average statistics data"),
            ("math", "Exponents",
             "An exponent shows how many times a number (the base) is "
             "multiplied by itself, e.g. 2^3 = 2 x 2 x 2 = 8.",
             "exponent power base multiply repeated"),
            ("math", "Types of Triangles",
             "Triangles are classified by sides (equilateral, isosceles, "
             "scalene) or by angles (acute, right, obtuse).",
             "triangle equilateral isosceles scalene acute right obtuse"),
            ("math", "Integers and Number Types",
             "Natural numbers (1,2,3...), whole numbers (0,1,2...), integers "
             "(...,-1,0,1...), rational numbers (fractions), and irrational "
             "numbers (like pi) make up the real number system.",
             "integer natural whole rational irrational number system"),

            # --- English / Grammar ---
            ("english", "Noun",
             "A noun is a word that names a person, place, thing, or idea, "
             "e.g. 'city', 'dog', 'freedom'.",
             "noun grammar part of speech name person place thing"),
            ("english", "Verb",
             "A verb expresses an action, occurrence, or state of being, "
             "e.g. 'run', 'become', 'is'.",
             "verb grammar action state part of speech"),
            ("english", "Simile vs Metaphor",
             "A simile compares two things using 'like' or 'as' (e.g. 'brave "
             "as a lion'). A metaphor states one thing IS another (e.g. 'he is "
             "a lion').",
             "simile metaphor comparison figurative language literary device"),
            ("english", "Homophones",
             "Homophones are words that sound alike but have different meanings "
             "and often different spellings, e.g. 'their', 'there', 'they're'.",
             "homophone spelling sound words english"),
            ("english", "Active vs Passive Voice",
             "In active voice the subject performs the action ('She wrote the "
             "letter'). In passive voice the subject receives the action "
             "('The letter was written by her').",
             "active passive voice grammar sentence structure"),
            ("english", "Adjective",
             "An adjective describes or modifies a noun, giving more detail "
             "about size, color, quality, or quantity, e.g. 'tall', 'blue', "
             "'three'.",
             "adjective describe noun modify grammar"),
            ("english", "Adverb",
             "An adverb modifies a verb, adjective, or another adverb, often "
             "answering how, when, where, or to what degree — many end in "
             "'-ly' (e.g. 'quickly').",
             "adverb modify verb grammar ly"),
            ("english", "Synonyms and Antonyms",
             "Synonyms are words with similar meanings (e.g. 'happy' and "
             "'joyful'). Antonyms are words with opposite meanings (e.g. "
             "'happy' and 'sad').",
             "synonym antonym meaning opposite similar words"),
            ("english", "Punctuation Basics",
             "Periods end statements, question marks end questions, commas "
             "separate items or clauses, and apostrophes show possession or "
             "contractions (e.g. 'it's', 'dog's').",
             "punctuation period comma apostrophe question mark grammar"),
            ("english", "Types of Sentences",
             "Sentences can be declarative (statement), interrogative "
             "(question), imperative (command), or exclamatory (strong "
             "feeling).",
             "sentence type declarative interrogative imperative exclamatory"),
            ("english", "Subject and Predicate",
             "Every complete sentence has a subject (who or what it's about) "
             "and a predicate (what the subject does or is), e.g. 'The dog "
             "[subject] barked loudly [predicate].'",
             "subject predicate sentence structure grammar"),
            ("english", "Prefixes and Suffixes",
             "A prefix is added to the start of a word to change its meaning "
             "(e.g. 'un-' + happy = unhappy). A suffix is added to the end "
             "(e.g. happy + '-ness' = happiness).",
             "prefix suffix word root meaning affix"),

            # --- General ---
            ("general", "Python Programming Language",
             "Python is a high-level, interpreted programming language known "
             "for readable syntax, created by Guido van Rossum, first released "
             "in 1991.",
             "python programming language guido van rossum"),
            ("general", "Continents",
             "There are seven continents: Africa, Antarctica, Asia, Australia, "
             "Europe, North America, and South America.",
             "continents world geography africa asia europe"),
            ("general", "Human Heart",
             "The human heart has four chambers (two atria, two ventricles) "
             "and pumps blood through the circulatory system.",
             "heart chambers blood circulatory human body"),
            ("general", "Seven Wonders of the Ancient World",
             "The classic Seven Wonders of the Ancient World were the Great "
             "Pyramid of Giza, the Hanging Gardens of Babylon, the Statue of "
             "Zeus at Olympia, the Temple of Artemis, the Mausoleum at "
             "Halicarnassus, the Colossus of Rhodes, and the Lighthouse of "
             "Alexandria.",
             "seven wonders ancient world pyramid giza history"),
            ("general", "World's Longest River",
             "The Nile and the Amazon are generally considered the two "
             "longest rivers in the world, with measurements historically "
             "putting the Nile slightly ahead at roughly 6,650 km.",
             "longest river nile amazon geography world"),
            ("general", "Largest Ocean",
             "The Pacific Ocean is the largest and deepest ocean on Earth, "
             "covering roughly one-third of the planet's surface.",
             "largest ocean pacific geography world"),
        ]
        for category, topic, content, keywords in raw_entries:
            self._insert(category, topic, content, keywords)

        # Bulk-generated reference entries (each is individually accurate,
        # generated from data tables rather than typed out by hand, so the
        # knowledge base can comfortably clear 100+ entries per category).
        for category, topic, content, keywords in self._generate_periodic_table_entries():
            self._insert(category, topic, content, keywords)
        for category, topic, content, keywords in self._generate_math_reference_entries():
            self._insert(category, topic, content, keywords)
        for category, topic, content, keywords in self._generate_english_reference_entries():
            self._insert(category, topic, content, keywords)
        for category, topic, content, keywords in self._generate_world_capitals_entries():
            self._insert(category, topic, content, keywords)

    # ---------- bulk reference-data generators ----------

    @staticmethod
    def _generate_periodic_table_entries() -> List[Tuple[str, str, str, str]]:
        """All 118 known chemical elements, generated from a data table
        rather than hand-typed, so each one is consistent and accurate."""
        elements = [
            (1, "H", "Hydrogen"), (2, "He", "Helium"), (3, "Li", "Lithium"),
            (4, "Be", "Beryllium"), (5, "B", "Boron"), (6, "C", "Carbon"),
            (7, "N", "Nitrogen"), (8, "O", "Oxygen"), (9, "F", "Fluorine"),
            (10, "Ne", "Neon"), (11, "Na", "Sodium"), (12, "Mg", "Magnesium"),
            (13, "Al", "Aluminium"), (14, "Si", "Silicon"), (15, "P", "Phosphorus"),
            (16, "S", "Sulfur"), (17, "Cl", "Chlorine"), (18, "Ar", "Argon"),
            (19, "K", "Potassium"), (20, "Ca", "Calcium"), (21, "Sc", "Scandium"),
            (22, "Ti", "Titanium"), (23, "V", "Vanadium"), (24, "Cr", "Chromium"),
            (25, "Mn", "Manganese"), (26, "Fe", "Iron"), (27, "Co", "Cobalt"),
            (28, "Ni", "Nickel"), (29, "Cu", "Copper"), (30, "Zn", "Zinc"),
            (31, "Ga", "Gallium"), (32, "Ge", "Germanium"), (33, "As", "Arsenic"),
            (34, "Se", "Selenium"), (35, "Br", "Bromine"), (36, "Kr", "Krypton"),
            (37, "Rb", "Rubidium"), (38, "Sr", "Strontium"), (39, "Y", "Yttrium"),
            (40, "Zr", "Zirconium"), (41, "Nb", "Niobium"), (42, "Mo", "Molybdenum"),
            (43, "Tc", "Technetium"), (44, "Ru", "Ruthenium"), (45, "Rh", "Rhodium"),
            (46, "Pd", "Palladium"), (47, "Ag", "Silver"), (48, "Cd", "Cadmium"),
            (49, "In", "Indium"), (50, "Sn", "Tin"), (51, "Sb", "Antimony"),
            (52, "Te", "Tellurium"), (53, "I", "Iodine"), (54, "Xe", "Xenon"),
            (55, "Cs", "Caesium"), (56, "Ba", "Barium"), (57, "La", "Lanthanum"),
            (58, "Ce", "Cerium"), (59, "Pr", "Praseodymium"), (60, "Nd", "Neodymium"),
            (61, "Pm", "Promethium"), (62, "Sm", "Samarium"), (63, "Eu", "Europium"),
            (64, "Gd", "Gadolinium"), (65, "Tb", "Terbium"), (66, "Dy", "Dysprosium"),
            (67, "Ho", "Holmium"), (68, "Er", "Erbium"), (69, "Tm", "Thulium"),
            (70, "Yb", "Ytterbium"), (71, "Lu", "Lutetium"), (72, "Hf", "Hafnium"),
            (73, "Ta", "Tantalum"), (74, "W", "Tungsten"), (75, "Re", "Rhenium"),
            (76, "Os", "Osmium"), (77, "Ir", "Iridium"), (78, "Pt", "Platinum"),
            (79, "Au", "Gold"), (80, "Hg", "Mercury"), (81, "Tl", "Thallium"),
            (82, "Pb", "Lead"), (83, "Bi", "Bismuth"), (84, "Po", "Polonium"),
            (85, "At", "Astatine"), (86, "Rn", "Radon"), (87, "Fr", "Francium"),
            (88, "Ra", "Radium"), (89, "Ac", "Actinium"), (90, "Th", "Thorium"),
            (91, "Pa", "Protactinium"), (92, "U", "Uranium"), (93, "Np", "Neptunium"),
            (94, "Pu", "Plutonium"), (95, "Am", "Americium"), (96, "Cm", "Curium"),
            (97, "Bk", "Berkelium"), (98, "Cf", "Californium"), (99, "Es", "Einsteinium"),
            (100, "Fm", "Fermium"), (101, "Md", "Mendelevium"), (102, "No", "Nobelium"),
            (103, "Lr", "Lawrencium"), (104, "Rf", "Rutherfordium"), (105, "Db", "Dubnium"),
            (106, "Sg", "Seaborgium"), (107, "Bh", "Bohrium"), (108, "Hs", "Hassium"),
            (109, "Mt", "Meitnerium"), (110, "Ds", "Darmstadtium"), (111, "Rg", "Roentgenium"),
            (112, "Cn", "Copernicium"), (113, "Nh", "Nihonium"), (114, "Fl", "Flerovium"),
            (115, "Mc", "Moscovium"), (116, "Lv", "Livermorium"), (117, "Ts", "Tennessine"),
            (118, "Og", "Oganesson"),
        ]
        entries = []
        for number, symbol, name in elements:
            entries.append((
                "science",
                f"Element: {name}",
                f"{name} (symbol {symbol}) is element number {number} on the "
                f"periodic table.",
                f"{name.lower()} {symbol.lower()} element {number} periodic table atom chemistry",
            ))
        return entries

    @staticmethod
    def _generate_math_reference_entries() -> List[Tuple[str, str, str, str]]:
        """Squares, cubes, square roots, and Roman numerals — generated
        from simple arithmetic so every value is guaranteed correct."""
        entries = []

        for n in range(1, 41):
            entries.append((
                "math", f"Square of {n}",
                f"The square of {n} is {n * n} (that is, {n} x {n} = {n * n}).",
                f"square {n} squared {n * n} multiply power",
            ))

        for n in range(1, 16):
            entries.append((
                "math", f"Cube of {n}",
                f"The cube of {n} is {n ** 3} (that is, {n} x {n} x {n} = {n ** 3}).",
                f"cube {n} cubed {n ** 3} multiply power",
            ))

        for n in range(1, 21):
            square = n * n
            entries.append((
                "math", f"Square Root of {square}",
                f"The square root of {square} is {n}.",
                f"square root sqrt {square} {n}",
            ))

        roman_table = [
            (1000, "M"), (900, "CM"), (500, "D"), (400, "CD"),
            (100, "C"), (90, "XC"), (50, "L"), (40, "XL"),
            (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I"),
        ]

        def to_roman(num: int) -> str:
            result = ""
            for value, symbol in roman_table:
                while num >= value:
                    result += symbol
                    num -= value
            return result

        for n in range(1, 31):
            roman = to_roman(n)
            entries.append((
                "math", f"Roman Numeral for {n}",
                f"The Roman numeral for {n} is {roman}.",
                f"roman numeral {n} {roman.lower()}",
            ))

        return entries

    @staticmethod
    def _generate_english_reference_entries() -> List[Tuple[str, str, str, str]]:
        """Irregular verbs, homophone sets, contractions, and prefixes —
        generated from small reference tables."""
        entries = []

        irregular_verbs = [
            ("go", "went", "gone"), ("eat", "ate", "eaten"), ("see", "saw", "seen"),
            ("take", "took", "taken"), ("give", "gave", "given"), ("know", "knew", "known"),
            ("come", "came", "come"), ("run", "ran", "run"), ("do", "did", "done"),
            ("say", "said", "said"), ("get", "got", "gotten"), ("make", "made", "made"),
            ("find", "found", "found"), ("think", "thought", "thought"), ("tell", "told", "told"),
            ("become", "became", "become"), ("leave", "left", "left"), ("feel", "felt", "felt"),
            ("bring", "brought", "brought"), ("begin", "began", "begun"), ("keep", "kept", "kept"),
            ("hold", "held", "held"), ("write", "wrote", "written"), ("stand", "stood", "stood"),
            ("hear", "heard", "heard"), ("let", "let", "let"), ("mean", "meant", "meant"),
            ("set", "set", "set"), ("meet", "met", "met"), ("pay", "paid", "paid"),
            ("sit", "sat", "sat"), ("speak", "spoke", "spoken"), ("lie", "lay", "lain"),
            ("lead", "led", "led"), ("read", "read", "read"), ("grow", "grew", "grown"),
            ("lose", "lost", "lost"), ("fall", "fell", "fallen"), ("send", "sent", "sent"),
            ("build", "built", "built"), ("understand", "understood", "understood"),
            ("draw", "drew", "drawn"), ("break", "broke", "broken"), ("spend", "spent", "spent"),
            ("cut", "cut", "cut"), ("rise", "rose", "risen"), ("drive", "drove", "driven"),
            ("buy", "bought", "bought"), ("wear", "wore", "worn"), ("choose", "chose", "chosen"),
        ]
        for base, past, participle in irregular_verbs:
            entries.append((
                "english", f"Irregular Verb: {base}",
                f"The past tense of '{base}' is '{past}', and the past "
                f"participle is '{participle}'.",
                f"{base} {past} {participle} irregular verb tense grammar",
            ))

        homophone_sets = [
            ("to, too, two", "'To' is a preposition or infinitive marker, "
             "'too' means 'also' or 'excessively', and 'two' is the number 2."),
            ("their, there, they're", "'Their' shows possession, 'there' "
             "indicates a place, and 'they're' is a contraction of 'they are'."),
            ("your, you're", "'Your' shows possession; 'you're' is a "
             "contraction of 'you are'."),
            ("its, it's", "'Its' shows possession; 'it's' is a contraction "
             "of 'it is' or 'it has'."),
            ("here, hear", "'Here' refers to a place; 'hear' means to "
             "perceive sound."),
            ("write, right", "'Write' means to put words on paper; 'right' "
             "can mean correct or a direction."),
            ("break, brake", "'Break' means to separate into pieces; "
             "'brake' is the device used to stop a vehicle."),
            ("peace, piece", "'Peace' means calm or absence of conflict; "
             "'piece' means a part of something."),
            ("flower, flour", "'Flower' is a plant's bloom; 'flour' is a "
             "powder used in baking."),
            ("night, knight", "'Night' is the time when it's dark; "
             "'knight' is a medieval soldier of noble rank."),
            ("weather, whether", "'Weather' refers to atmospheric "
             "conditions; 'whether' introduces a choice between alternatives."),
            ("principal, principle", "'Principal' can mean the head of a "
             "school or most important; 'principle' means a fundamental "
             "rule or belief."),
            ("stationary, stationery", "'Stationary' means not moving; "
             "'stationery' refers to writing materials."),
            ("compliment, complement", "'Compliment' is praise; "
             "'complement' means something that completes or goes well "
             "with another."),
            ("dessert, desert", "'Dessert' is a sweet course after a meal; "
             "'desert' is a dry, arid region (or, as a verb, to abandon)."),
            ("board, bored", "'Board' can mean a flat piece of wood or a "
             "group of decision-makers; 'bored' means feeling weary or "
             "uninterested."),
            ("cell, sell", "'Cell' is a small room or a biological unit; "
             "'sell' means to exchange something for money."),
            ("mail, male", "'Mail' refers to letters and packages; 'male' "
             "refers to the sex that produces sperm."),
            ("sea, see", "'Sea' is a large body of salt water; 'see' means "
             "to perceive with the eyes."),
            ("blue, blew", "'Blue' is a color; 'blew' is the past tense of "
             "'blow'."),
        ]
        for words, explanation in homophone_sets:
            entries.append((
                "english", f"Homophones: {words}", explanation,
                f"homophone {words.replace(',', '')} spelling sound",
            ))

        contractions = [
            ("can't", "cannot"), ("won't", "will not"), ("don't", "do not"),
            ("doesn't", "does not"), ("didn't", "did not"), ("isn't", "is not"),
            ("aren't", "are not"), ("wasn't", "was not"), ("weren't", "were not"),
            ("haven't", "have not"), ("hasn't", "has not"), ("hadn't", "had not"),
            ("wouldn't", "would not"), ("shouldn't", "should not"), ("couldn't", "could not"),
            ("I'm", "I am"), ("you're", "you are"), ("he's", "he is"),
            ("she's", "she is"), ("it's", "it is"), ("we're", "we are"),
            ("they're", "they are"), ("I'll", "I will"), ("you'll", "you will"),
            ("he'll", "he will"), ("she'll", "she will"), ("we'll", "we will"),
            ("they'll", "they will"), ("I've", "I have"), ("you've", "you have"),
        ]
        for contraction, full in contractions:
            entries.append((
                "english", f"Contraction: {contraction}",
                f"'{contraction}' is the contraction of '{full}'.",
                f"{contraction} {full} contraction grammar apostrophe",
            ))

        prefixes = [
            ("un-", "not / the opposite of", "unhappy"),
            ("re-", "again", "rewrite"),
            ("pre-", "before", "preview"),
            ("dis-", "not / opposite of", "disagree"),
            ("mis-", "wrongly", "misunderstand"),
            ("non-", "not", "nonstop"),
            ("over-", "too much", "overcook"),
            ("under-", "too little", "underestimate"),
            ("sub-", "under or below", "submarine"),
            ("inter-", "between", "international"),
            ("super-", "above or beyond", "superhuman"),
            ("anti-", "against", "antisocial"),
            ("auto-", "self", "automatic"),
            ("bi-", "two", "bicycle"),
            ("tri-", "three", "triangle"),
            ("ex-", "former or out", "ex-president"),
            ("de-", "reverse or remove", "deactivate"),
            ("semi-", "half", "semicircle"),
            ("post-", "after", "postpone"),
            ("co-", "together", "cooperate"),
        ]
        for prefix, meaning, example in prefixes:
            entries.append((
                "english", f"Prefix: {prefix}",
                f"The prefix '{prefix}' means '{meaning}', as in '{example}'.",
                f"{prefix} prefix meaning {meaning} word root",
            ))

        return entries

    @staticmethod
    def _generate_world_capitals_entries() -> List[Tuple[str, str, str, str]]:
        """Country/capital pairs, limited to widely-agreed, uncontested
        capitals to keep the data straightforwardly factual."""
        capitals = [
            ("France", "Paris"), ("Germany", "Berlin"), ("Italy", "Rome"),
            ("Spain", "Madrid"), ("Portugal", "Lisbon"), ("United Kingdom", "London"),
            ("Ireland", "Dublin"), ("Netherlands", "Amsterdam"), ("Belgium", "Brussels"),
            ("Switzerland", "Bern"), ("Austria", "Vienna"), ("Sweden", "Stockholm"),
            ("Norway", "Oslo"), ("Denmark", "Copenhagen"), ("Finland", "Helsinki"),
            ("Iceland", "Reykjavik"), ("Poland", "Warsaw"), ("Czech Republic", "Prague"),
            ("Slovakia", "Bratislava"), ("Hungary", "Budapest"), ("Romania", "Bucharest"),
            ("Bulgaria", "Sofia"), ("Greece", "Athens"), ("Turkey", "Ankara"),
            ("Russia", "Moscow"), ("Ukraine", "Kyiv"), ("Belarus", "Minsk"),
            ("Croatia", "Zagreb"), ("Serbia", "Belgrade"), ("Slovenia", "Ljubljana"),
            ("Estonia", "Tallinn"), ("Latvia", "Riga"), ("Lithuania", "Vilnius"),
            ("Albania", "Tirana"), ("North Macedonia", "Skopje"), ("Moldova", "Chisinau"),
            ("Georgia", "Tbilisi"), ("Armenia", "Yerevan"), ("Azerbaijan", "Baku"),
            ("Kazakhstan", "Astana"), ("Uzbekistan", "Tashkent"), ("China", "Beijing"),
            ("Japan", "Tokyo"), ("South Korea", "Seoul"), ("North Korea", "Pyongyang"),
            ("India", "New Delhi"), ("Pakistan", "Islamabad"), ("Bangladesh", "Dhaka"),
            ("Nepal", "Kathmandu"), ("Bhutan", "Thimphu"), ("Myanmar", "Naypyidaw"),
            ("Thailand", "Bangkok"), ("Vietnam", "Hanoi"), ("Cambodia", "Phnom Penh"),
            ("Laos", "Vientiane"), ("Malaysia", "Kuala Lumpur"), ("Singapore", "Singapore"),
            ("Indonesia", "Jakarta"), ("Philippines", "Manila"), ("Mongolia", "Ulaanbaatar"),
            ("Afghanistan", "Kabul"), ("Iran", "Tehran"), ("Iraq", "Baghdad"),
            ("Saudi Arabia", "Riyadh"), ("United Arab Emirates", "Abu Dhabi"), ("Qatar", "Doha"),
            ("Kuwait", "Kuwait City"), ("Jordan", "Amman"), ("Lebanon", "Beirut"),
            ("Syria", "Damascus"), ("Oman", "Muscat"), ("Egypt", "Cairo"),
            ("Libya", "Tripoli"), ("Tunisia", "Tunis"), ("Algeria", "Algiers"),
            ("Morocco", "Rabat"), ("Sudan", "Khartoum"), ("Ethiopia", "Addis Ababa"),
            ("Kenya", "Nairobi"), ("Tanzania", "Dodoma"), ("Uganda", "Kampala"),
            ("Rwanda", "Kigali"), ("Nigeria", "Abuja"), ("Ghana", "Accra"),
            ("Senegal", "Dakar"), ("Ivory Coast", "Yamoussoukro"), ("Cameroon", "Yaounde"),
            ("Democratic Republic of the Congo", "Kinshasa"), ("Angola", "Luanda"),
            ("Zambia", "Lusaka"), ("Zimbabwe", "Harare"), ("Mozambique", "Maputo"),
            ("Namibia", "Windhoek"), ("Botswana", "Gaborone"), ("Madagascar", "Antananarivo"),
            ("Australia", "Canberra"), ("New Zealand", "Wellington"), ("Canada", "Ottawa"),
            ("United States", "Washington, D.C."), ("Mexico", "Mexico City"),
            ("Brazil", "Brasilia"), ("Argentina", "Buenos Aires"), ("Chile", "Santiago"),
            ("Peru", "Lima"), ("Colombia", "Bogota"), ("Venezuela", "Caracas"),
            ("Ecuador", "Quito"), ("Bolivia", "Sucre"), ("Uruguay", "Montevideo"),
            ("Paraguay", "Asuncion"), ("Cuba", "Havana"), ("Jamaica", "Kingston"),
            ("Panama", "Panama City"), ("Costa Rica", "San Jose"), ("Guatemala", "Guatemala City"),
            ("Honduras", "Tegucigalpa"), ("Nicaragua", "Managua"), ("El Salvador", "San Salvador"),
        ]
        entries = []
        for country, capital in capitals:
            entries.append((
                "general", f"Capital of {country}",
                f"The capital of {country} is {capital}.",
                f"{country.lower()} capital {capital.lower()} geography country",
            ))
        return entries

    def _insert(self, category: str, topic: str, content: str, keywords: str,
                user_taught: bool = False) -> int:
        entry_id = self._next_id
        self._next_id += 1
        self.entries.append({
            "id": entry_id,
            "category": category,
            "topic": topic,
            "content": content,
            "keywords": keywords,
            "user_taught": user_taught,
        })
        return entry_id

    # ---------- public API ----------

    def add_entry(self, category: str, topic: str, content: str, keywords: str = "") -> int:
        """Add a new fact to the knowledge base. Returns the new entry id."""
        if not keywords:
            keywords = " ".join(self._tokenize(topic + " " + content))
        entry_id = self._insert(
            category.lower().strip(), topic.strip(), content.strip(),
            keywords.lower(), user_taught=True,
        )
        self._build_idf_index()  # keep search weighting up to date
        return entry_id

    def forget(self, topic: str) -> List[str]:
        """
        Removes any user-taught entry (added via 'remember X is Y') whose
        topic matches the given text, either exactly or as a close/partial
        match. Built-in knowledge-base facts can never be forgotten this
        way — only things the user taught during this session. Returns the
        list of topic names that were removed.
        """
        needle = topic.strip().lower()
        if not needle:
            return []

        removed_topics: List[str] = []
        remaining: List[dict] = []
        for entry in self.entries:
            is_match = (
                entry["user_taught"]
                and (
                    entry["topic"].strip().lower() == needle
                    or needle in entry["topic"].strip().lower()
                    or entry["topic"].strip().lower() in needle
                )
            )
            if is_match:
                removed_topics.append(entry["topic"])
            else:
                remaining.append(entry)

        if removed_topics:
            self.entries = remaining
            self._build_idf_index()
        return removed_topics

    # Common words that appear everywhere and shouldn't drive relevance
    # scoring on their own (e.g. "is" showing up in "he is" -> "he's").
    # This also includes conversational filler ("tell", "me", "about",
    # "please"...) — without excluding these, an unseen filler word would
    # get the *maximum* IDF weight (the same treatment reserved for rare,
    # meaningful words) and unfairly drown out a genuine match, e.g. "tell
    # me about photosynthesis" scoring worse than plain "photosynthesis".
    _STOPWORDS = {
        "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
        "of", "to", "in", "on", "at", "for", "and", "or", "but", "with",
        "what", "who", "whom", "which", "that", "this", "these", "those",
        "do", "does", "did", "how", "why", "when", "where", "it", "its",
        "as", "by", "from", "so", "if", "than", "then", "there", "here",
        "tell", "me", "about", "please", "can", "could", "would", "give",
        "know", "explain", "describe", "define", "us", "you", "i", "we",
        "let", "want", "need", "like", "some", "any", "just", "quick",
        "briefly", "brief", "info", "information", "up", "out",
    }

    def _build_idf_index(self) -> None:
        """
        Computes an inverse-document-frequency weight for every token seen
        across the knowledge base. A word that appears in only one or two
        entries (like "photosynthesis" or "gold") gets a high weight — a
        match on it is meaningful. A word that appears in dozens of entries
        because it's a shared label for a whole batch of generated facts
        (like "capital" across 118 country entries, or "grammar" across
        dozens of English entries) gets a low weight, so it alone can't
        make an unrelated entry look relevant. This is what stops a query
        like "capital of Atlantis" from confidently matching "Capital of
        Iran" just because both mention the word "capital".
        """
        doc_count: dict = {}
        total_docs = len(self.entries)
        for entry in self.entries:
            tokens = set(
                self._tokenize(entry["keywords"] + " " + entry["topic"])
            ) - self._STOPWORDS
            for token in tokens:
                doc_count[token] = doc_count.get(token, 0) + 1

        self._idf = {
            token: math.log((total_docs + 1) / (freq + 1)) + 1.0
            for token, freq in doc_count.items()
        }
        # A token never seen in the knowledge base is treated as at least
        # as rare/distinctive as the rarest known token.
        self._default_idf = max(self._idf.values(), default=1.0)

    def _weighted_overlap(self, query_tokens: set, entry_tokens: set) -> float:
        """
        Fraction (0..1) of the query's total IDF "weight" that is covered
        by tokens shared with this entry. 1.0 means every meaningful word
        in the query matched something in the entry; a low value means
        only generic, corpus-wide words matched.
        """
        if not query_tokens:
            return 0.0
        query_weight = sum(self._idf.get(t, self._default_idf) for t in query_tokens)
        if query_weight <= 0:
            return 0.0
        shared = query_tokens & entry_tokens
        shared_weight = sum(self._idf.get(t, self._default_idf) for t in shared)
        return shared_weight / query_weight

    def search(self, query: str, top_k: int = 3) -> List[Tuple[float, dict]]:
        """
        Rank every knowledge-base entry against the query using a blend of:
          - IDF-weighted keyword overlap (stopwords excluded, and common
            corpus-wide words downweighted, so they can't fake a match)
          - fuzzy string similarity (difflib) on topic and content
        Returns a list of (score, entry) tuples sorted best-first.

        A result is only accepted if it shows *genuine* similarity —
        either most of the query's meaningful words are covered by the
        entry, or there's a strong fuzzy match on the topic name (to catch
        typos like "fotosinthesis"). This stops random, nonsense, or
        near-miss input (like a country not in the data) from matching an
        unrelated entry just because they share one generic word.
        """
        query_tokens = set(self._tokenize(query)) - self._STOPWORDS
        if not query_tokens:
            return []

        scored: List[Tuple[float, dict]] = []

        for entry in self.entries:
            entry_tokens = set(
                self._tokenize(entry["keywords"] + " " + entry["topic"])
            ) - self._STOPWORDS
            overlap_score = self._weighted_overlap(query_tokens, entry_tokens)

            fuzzy_topic = difflib.SequenceMatcher(
                None, query.lower(), entry["topic"].lower()
            ).ratio()
            fuzzy_content = difflib.SequenceMatcher(
                None, query.lower(), entry["content"].lower()
            ).ratio()

            score = (overlap_score * 2.0) + (fuzzy_topic * 1.0) + (fuzzy_content * 0.5)

            # Per-token fuzzy check (for catching a misspelled keyword, e.g.
            # "fotosinthesis" vs "photosynthesis"). Only compares words that
            # did NOT already overlap exactly — otherwise a shared generic
            # word (like "capital" appearing in every "Capital of X" entry)
            # would trivially score a perfect 1.0 fuzzy match against
            # itself and defeat the whole point of this check. This is also
            # why whole-string fuzzy matching isn't used here: many
            # generated entries share a template prefix ("Capital of ",
            # "Contraction: ", "Element: "), which makes full-string
            # fuzzy matching unreliable — "capital of atlantis" looks very
            # similar to "Capital of Albania" as strings even though the
            # countries have nothing to do with each other.
            unmatched_query = query_tokens - entry_tokens
            unmatched_entry = entry_tokens - query_tokens
            token_fuzzy = max(
                (
                    difflib.SequenceMatcher(None, q_tok, e_tok).ratio()
                    for q_tok in unmatched_query
                    for e_tok in unmatched_entry
                ),
                default=0.0,
            )

            genuinely_relevant = overlap_score >= 0.5 or token_fuzzy >= 0.8
            if score > 0.5 and genuinely_relevant:
                scored.append((score, entry))

        scored.sort(key=lambda pair: pair[0], reverse=True)
        return scored[:top_k]

    def best_match_coverage(self, query: str) -> Tuple[Optional[dict], float]:
        """
        Finds the single entry that covers the most of the query's
        meaningful (IDF-weighted) content, and returns how much of the
        query that entry covers, from 0.0 (nothing) to 1.0 (all of it).

        Used to tell whether a query is really "about one topic" (a high
        coverage score, even if that topic's name contains connector words
        like "and") versus genuinely asking about several different things.
        """
        query_tokens = set(self._tokenize(query)) - self._STOPWORDS
        if not query_tokens:
            return None, 0.0

        best_entry: Optional[dict] = None
        best_overlap = 0.0
        for entry in self.entries:
            entry_tokens = set(
                self._tokenize(entry["keywords"] + " " + entry["topic"])
            ) - self._STOPWORDS
            overlap = self._weighted_overlap(query_tokens, entry_tokens)
            if overlap > best_overlap:
                best_overlap = overlap
                best_entry = entry
        return best_entry, best_overlap

    def count(self) -> int:
        return len(self.entries)

    def category_breakdown(self) -> dict:
        breakdown: dict = {}
        for entry in self.entries:
            breakdown[entry["category"]] = breakdown.get(entry["category"], 0) + 1
        return breakdown

    @staticmethod
    def _tokenize(text: str) -> List[str]:
        return re.findall(r"[a-z0-9]+", text.lower())


# ============================================================
# 2. SAFE MATH EVALUATOR
# ============================================================

class SafeMathEvaluator:
    """
    Evaluates arithmetic expressions safely using Python's `ast` module
    instead of calling eval() directly on user input.
    Supports + - * / // % ** and common functions: sqrt, sin, cos, tan,
    log, factorial, abs, round.
    """

    _BIN_OPS = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.FloorDiv: operator.floordiv,
        ast.Mod: operator.mod,
        ast.Pow: operator.pow,
    }
    _UNARY_OPS = {
        ast.UAdd: operator.pos,
        ast.USub: operator.neg,
    }
    _FUNCS = {
        "sqrt": math.sqrt,
        "sin": math.sin,
        "cos": math.cos,
        "tan": math.tan,
        "log": math.log,
        "log10": math.log10,
        "factorial": math.factorial,
        "abs": abs,
        "round": round,
    }
    _CONSTANTS = {"pi": math.pi, "e": math.e}

    @classmethod
    def looks_like_math(cls, text: str) -> bool:
        cleaned = text.lower()
        has_symbol = bool(re.search(
            r"[\d][\s]*[\+\-\*/\^%]|sqrt|factorial|log\(|sin\(|cos\(|tan\(|square root of",
            cleaned,
        ))
        has_digit = bool(re.search(r"\d", cleaned))
        return has_digit and has_symbol

    @classmethod
    def evaluate(cls, expression: str) -> Optional[float]:
        expr = expression.replace("^", "**").strip()
        expr = expr.rstrip("?").strip()
        expr = cls._normalize_phrasing(expr)
        try:
            tree = ast.parse(expr, mode="eval")
            return cls._eval_node(tree.body)
        except Exception:
            return None

    @staticmethod
    def _normalize_phrasing(expr: str) -> str:
        """Turn common plain-English math phrasing into valid syntax, e.g.
        '5 factorial' -> 'factorial(5)', 'square root of 81' -> 'sqrt(81)'."""
        expr = re.sub(
            r"(?:the\s+)?square root of\s+(-?\d+(?:\.\d+)?)", r"sqrt(\1)", expr
        )
        expr = re.sub(
            r"(-?\d+(?:\.\d+)?)\s+factorial", r"factorial(\1)", expr
        )
        expr = re.sub(
            r"factorial of\s+(-?\d+(?:\.\d+)?)", r"factorial(\1)", expr
        )
        return expr

    @classmethod
    def _eval_node(cls, node):
        if isinstance(node, ast.Constant):
            if isinstance(node.value, (int, float)):
                return node.value
            raise ValueError("Unsupported constant")
        if isinstance(node, ast.BinOp) and type(node.op) in cls._BIN_OPS:
            left = cls._eval_node(node.left)
            right = cls._eval_node(node.right)
            return cls._BIN_OPS[type(node.op)](left, right)
        if isinstance(node, ast.UnaryOp) and type(node.op) in cls._UNARY_OPS:
            return cls._UNARY_OPS[type(node.op)](cls._eval_node(node.operand))
        if isinstance(node, ast.Call):
            fname = getattr(node.func, "id", None)
            if fname in cls._FUNCS:
                args = [cls._eval_node(a) for a in node.args]
                return cls._FUNCS[fname](*args)
            raise ValueError(f"Unsupported function: {fname}")
        if isinstance(node, ast.Name) and node.id in cls._CONSTANTS:
            return cls._CONSTANTS[node.id]
        raise ValueError(f"Unsupported expression: {ast.dump(node)}")


# ============================================================
# 3. CONVERSATION / SMALL TALK LAYER
# ============================================================

class SmallTalk:
    """
    Pattern-based responses for casual conversation.

    Each pattern maps to a pool of possible replies. To avoid sounding
    robotic, this class remembers the last reply it gave for each
    pattern and avoids repeating it back-to-back (it picks randomly
    among the *other* options first, only reusing a line once every
    other option in that pool has been tried).
    """

    PATTERNS: List[Tuple[str, List[str]]] = [
        # --- Greetings ---
        (r"\b(hi|hello|hey|yo|greetings|howdy|sup)\b",
         ["Hello! What can I help you with today?",
          "Hi there! Ask me a question or search my knowledge base.",
          "Hey! I'm ready when you are.",
          "Hiya! What's on your mind?",
          "Hello again! Fire away with a question.",
          "Hey there — science, math, English, or just a chat, I'm in."]),

        # --- How are you / wellbeing ---
        (r"\bhow are you\b|\bhow('?s| is) it going\b|\bhow do you feel\b",
         ["I'm running smoothly, thanks for asking! How about you?",
          "Doing great — all systems operational. How's your day going?",
          "Can't complain, I'm just a program, but I'm happy to chat!",
          "All good on my end! What's up with you?"]),
        (r"\b(i'?m|i am) (good|fine|great|ok|okay|alright)\b",
         ["Glad to hear it!", "That's great to hear.",
          "Nice, glad things are going well!"]),
        (r"\b(i'?m|i am) (sad|tired|bored|stressed|upset|not (great|good|feeling great))\b",
         ["Sorry to hear that. Want to take your mind off it with a fun fact "
          "or a quick math puzzle?",
          "I hear you. Sometimes a change of topic helps — ask me something "
          "random and I'll do my best.",
          "That's tough. I'm just a chatbot, but I'm happy to keep you "
          "company — ask me anything."]),

        # --- Identity ---
        (r"\bwhat('?s| is) your name\b|\bwho are you\b",
         ["I'm ZAI — a knowledge search and Q&A assistant.",
          "You can call me ZAI. I search a knowledge base and answer questions.",
          "I go by ZAI — made by ZXStudios to help answer your questions."]),
        (r"\bwho (made|created|built) you\b",
         ["I was built by ZXStudios — a custom Python program that searches a "
          "knowledge base and chats a little on the side.",
          "ZXStudios made me! I'm a Python program under the hood."]),
        (r"\bhow old are you\b|\bwhat('?s| is) your age\b",
         ["I don't have an age in the human sense — I'm just code that runs "
          "whenever you start the program!",
          "Ageless, really — I only exist while this script is running."]),
        (r"\bare you (real|human|alive|a robot|an ai|sentient)\b",
         ["I'm a program, not a person — just pattern matching and a "
          "knowledge base, no consciousness involved!",
          "Nope, no consciousness here — just Python code doing its best."]),

        # --- Capabilities ---
        (r"\bwhat (can|do) you do\b|\bhelp me\b|\bwhat are you (good|useful) for\b",
         ["I can search a knowledge base of science, math, and English facts, "
          "solve math expressions, and hold a simple conversation.",
          "Ask me a science, math, or English question, give me a math "
          "expression to solve, or just chat — I'll do my best either way.",
          "My specialties are knowledge lookups and quick math — try me!"]),

        # --- Gratitude ---
        (r"\b(thanks|thank you|appreciate it|thx|ty)\b",
         ["You're welcome!", "Anytime!", "Happy to help.",
          "No problem at all!", "Glad I could help!"]),

        # --- Farewells ---
        (r"\b(bye|goodbye|see you|see ya|later|farewell)\b",
         ["Goodbye! Come back anytime.", "See you later!",
          "Take care — I'll be here when you're back.",
          "Bye for now!"]),

        # --- Jokes / fun ---
        (r"\bjoke\b|\bmake me laugh\b|\bsomething funny\b",
         ["Why did the function stop calling itself? It had too much recursion "
          "and no base case!",
          "There are 10 kinds of people: those who understand binary and "
          "those who don't.",
          "Why do programmers prefer dark mode? Because light attracts bugs.",
          "I told my computer I needed a break, and now it won't stop "
          "sending me KitKat ads.",
          "Why was the math book sad? It had too many problems."]),

        # --- Time / date ---
        (r"\btime\b|\bdate\b|\bwhat day is it\b",
         [f"Right now it's {datetime.now().strftime('%A, %B %d, %Y %H:%M')}."]),

        # --- Compliments / small reactions ---
        (r"\byou'?re (smart|cool|great|awesome|amazing|the best)\b",
         ["Aw, thanks! I try my best.", "That means a lot, thank you!",
          "You're pretty great yourself for saying that!"]),
        (r"\b(nice|cool|awesome|wow|great|sweet)\b",
         ["Glad you think so!", "Right?", "I thought so too.",
          "Cool, huh?"]),
        (r"\b(lol|haha|hehe|lmao)\b",
         ["Glad that landed!", "Ha! Got you.", "😄"]),

        # --- Yes/no acknowledgements ---
        (r"^\s*(ok|okay|alright|sure|fine)\s*[.!]?\s*$",
         ["Cool — what's next?", "Alright! Ask away whenever you're ready.",
          "Got it. What else can I help with?"]),
        (r"^\s*(yes|yep|yeah|yup)\s*[.!]?\s*$",
         ["Great, let's keep going.", "Awesome, what's next?"]),
        (r"^\s*(no|nope|nah)\s*[.!]?\s*$",
         ["No worries, let me know if you change your mind.",
          "Alright, what would you like to do instead?"]),

        # --- Follow-up prompts ---
        (r"\btell me more\b|\bgo on\b|\bcontinue\b",
         ["Ask me a specific question and I'll dig deeper into it — try "
          "naming a topic in science, math, or English.",
          "I'd love to! Give me a topic to expand on."]),
        (r"\bwhat('?s| is) up\b",
         ["Not much, just waiting to answer your next question!",
          "Same as always — ready to help. What's up with you?"]),

        # --- Time-of-day greetings ---
        (r"\bgood morning\b",
         ["Good morning! Hope your day's off to a good start.",
          "Morning! What can I help you with?"]),
        (r"\bgood afternoon\b",
         ["Good afternoon! What can I do for you?",
          "Afternoon! Ask away."]),
        (r"\bgood (evening|night)\b",
         ["Good evening! How can I help?",
          "Evening! What's on your mind?"]),

        # --- Apologies ---
        (r"\b(sorry|my bad|apologies|apologize)\b",
         ["No worries at all!", "All good, no need to apologize.",
          "It's totally fine!"]),

        # --- Confusion / clarification ---
        (r"\bi don'?t understand\b|\bwhat do you mean\b|\bcan you repeat\b|\bcan you explain that again\b",
         ["Sure — could you tell me which part you'd like me to go over again?",
          "No problem, let's slow down. What part is unclear?",
          "Happy to clarify — which bit should I explain differently?"]),
        (r"\bnever ?mind\b|\bforget it\b|\bit'?s ok(ay)? i'?ll figure it out\b",
         ["No worries, I'm here if you change your mind.",
          "All good — just say the word if you want to pick it back up."]),

        # --- Agreement / disagreement ---
        (r"\b(i agree|exactly|true|so true|makes sense|that'?s right|correct)\b",
         ["Glad we're on the same page!", "Right?", "Exactly!"]),
        (r"\b(i disagree|that'?s wrong|no way|really\??)\b",
         ["Fair enough — I'm always open to being corrected if I got "
          "something wrong.",
          "Interesting, tell me more about why you see it that way."]),

        # --- Curiosity / prompts for more ---
        (r"\bsurprise me\b|\btell me something interesting\b|\bfun fact\b|\bdid you know\b",
         ["Here's one: honey never spoils — archaeologists have found "
          "edible honey in ancient Egyptian tombs.",
          "Fun fact: an octopus has three hearts and blue blood.",
          "Did you know a bolt of lightning is roughly five times hotter "
          "than the surface of the sun?",
          "Here's a good one: bananas are berries, but strawberries "
          "technically aren't."]),
        (r"\bwhat'?s new\b|\banything interesting\b",
         ["Not much new with me, but I'm always ready to dig into a "
          "topic — got something in mind?",
          "Same old knowledge base, always ready to search it for you!"]),

        # --- Weather small talk (no live data, but still conversational) ---
        (r"\bweather\b|\bis it raining\b|\bis it sunny\b",
         ["I can't check live weather, but I can tell you how the water "
          "cycle or seasons work if that's useful!",
          "No live weather feed here, sorry — but ask me about clouds, "
          "rain, or climate and I'll do my best."]),

        # --- Being told the AI is wrong / insulted ---
        (r"\byou'?re (wrong|dumb|stupid|useless|bad|terrible)\b",
         ["Sorry about that — if I got something wrong, let me know what "
          "and I'll take another look.",
          "Fair criticism, I'm not perfect. What did I get wrong?"]),

        # --- Requests the AI can't do ---
        (r"\bsing\b|\bsing me a song\b",
         ["I can't sing, but I could tell you a fact, a joke, or help "
          "with a question instead!"]),
        (r"\bcan you dance\b",
         ["No moves here, just text — but I've got plenty of facts and "
          "jokes if you want those instead."]),

        # --- Homework / help framing ---
        (r"\bhelp (me )?with (my )?homework\b|\bcan you help with school\b",
         ["Sure — send me the question and I'll do my best, especially "
          "for science, math, or English topics."]),

        # --- Generic short acknowledgements ---
        (r"^\s*(hmm+|interesting|got it|i see|noted|good to know)\s*[.!]?\s*$",
         ["Glad that helped!", "Let me know if you want more detail.",
          "Cool, anything else on your mind?"]),
    ]

    # A pool of neutral, friendly filler replies for short conversational
    # messages that don't match any specific pattern above and don't look
    # like an actual question — e.g. "cool story", "haha nice one", "aww".
    # This keeps the conversation flowing naturally instead of defaulting
    # to a "that's not in my knowledge base" message for things that were
    # never meant to be factual questions in the first place.
    _GENERIC_FILLERS = [
        "Gotcha!", "I hear you.", "Noted!", "Fair point.",
        "Makes sense to me.", "I'm with you.", "Good to know!",
        "Ha, I like that.", "Interesting way to put it.",
    ]

    def __init__(self):
        # Remembers the last reply used per pattern so we don't repeat
        # the same line twice in a row for the same kind of message.
        self._last_response: dict = {}

    def match(self, text: str) -> Optional[str]:
        lowered = text.lower()
        for pattern, responses in self.PATTERNS:
            if re.search(pattern, lowered):
                return self._pick(pattern, responses)
        return None

    def generic_reply(self) -> str:
        """A neutral, friendly filler for casual remarks that don't match
        any specific pattern and don't look like a real question."""
        return self._pick("__generic__", self._GENERIC_FILLERS)

    def _pick(self, pattern_key: str, responses: List[str]) -> str:
        if len(responses) == 1:
            return responses[0]
        last = self._last_response.get(pattern_key)
        candidates = [r for r in responses if r != last] or responses
        choice = random.choice(candidates)
        self._last_response[pattern_key] = choice
        return choice


# ============================================================
# 4. THE ADVANCED AI — intent routing + orchestration
# ============================================================

class AdvancedAI:
    def __init__(self):
        self.db = KnowledgeDatabase()
        self.math_evaluator = SafeMathEvaluator()
        self.small_talk = SmallTalk()
        self.history: List[Tuple[str, str]] = []  # (user_input, response)
        self._last_fallback: Optional[str] = None

    def process(self, user_input: str) -> str:
        text = user_input.strip()
        if not text:
            return "Say something and I'll do my best to respond!"

        response = self._route(text)
        self.history.append((text, response))
        return response

    # ---------- intent routing ----------

    def _route(self, text: str) -> str:
        lowered = text.lower()

        # 1. Explicit command to teach the AI something new
        add_match = re.match(
            r"remember(?: that)?\s+(?P<topic>.+?)\s+is\s+(?P<content>.+)",
            lowered,
        )
        if add_match:
            topic = add_match.group("topic").strip()
            content = add_match.group("content").strip()
            self.db.add_entry("general", topic, content)
            return f"Got it — I'll remember that {topic} is {content}."

        # 1b. Explicit command to make the AI forget something it was taught
        forget_match = re.match(r"forget(?: that)?\s+(?P<topic>.+)", lowered)
        if forget_match:
            topic = forget_match.group("topic").strip()
            removed = self.db.forget(topic)
            if removed:
                names = ", ".join(removed)
                return f"Done — I've forgotten what I knew about {names}."
            return (
                f"I couldn't find anything I was taught about '{topic}' to "
                "forget. (I can only forget things you've taught me with "
                "'remember X is Y' — not my built-in knowledge.)"
            )

        # 2. Math expressions
        if self.math_evaluator.looks_like_math(lowered):
            result = self.math_evaluator.evaluate(lowered)
            if result is not None:
                if isinstance(result, float) and result.is_integer():
                    result = int(result)
                return f"The answer is {result}."

        # 3. Small talk
        smalltalk_reply = self.small_talk.match(lowered)
        # Only use smalltalk if the message is short/conversational
        # (avoids hijacking longer factual questions that happen to contain
        # a matched word, e.g. "what is the time complexity of quicksort")
        if smalltalk_reply and len(lowered.split()) <= 6:
            return smalltalk_reply

        # 4. Multi-topic detection: is this really a question about two or
        # more different things at once? (e.g. "what is gravity and dna")
        multi_reply = self._maybe_multi_topic_reply(text)
        if multi_reply:
            return multi_reply

        # 5. Knowledge base search (single best-matching topic)
        results = self.db.search(text, top_k=3)
        if results:
            best_score, best_row = results[0]
            reply_lines = [
                f"**{best_row['topic']}** ({best_row['category']}): {best_row['content']}"
            ]
            if len(results) > 1:
                extras = ", ".join(r["topic"] for _, r in results[1:])
                reply_lines.append(f"(Related topics: {extras})")
            return "\n".join(reply_lines)

        # 6. Fallback to small talk even for longer text, then give up gracefully
        if smalltalk_reply:
            return smalltalk_reply

        return self._fallback(text)

    # ---------- multi-topic detection ----------

    # Words/punctuation that typically join two separate ideas in a
    # question, e.g. "gravity and dna", "gravity, dna", "gravity as well
    # as dna". Used to split a question into candidate sub-topics.
    _TOPIC_CONNECTORS = re.compile(
        r"\s*,\s*|\s+and\s+|\s+&\s+|\s+as well as\s+|\s+plus\s+|\s+also\s+",
        re.IGNORECASE,
    )

    def _maybe_multi_topic_reply(self, text: str) -> Optional[str]:
        """
        If the question is really about two or more distinct topics at
        once, returns a combined "here's both/all of them" reply. Returns
        None if the question is about a single topic (even one whose name
        happens to contain a word like "and", e.g. "Active vs Passive
        Voice" or "Mean, Median, and Mode") or if fewer than two confident
        topic matches are found.
        """
        # If one single entry already covers most of the query's content,
        # this is one topic, not several — even if its name contains a
        # connector word. Let the normal single-topic search handle it.
        best_entry, coverage = self.db.best_match_coverage(text)
        if best_entry and coverage >= 0.75:
            return None

        fragments = [f.strip() for f in self._TOPIC_CONNECTORS.split(text) if f.strip()]
        if len(fragments) < 2:
            return None

        matched_entries: List[dict] = []
        seen_topics = set()
        for fragment in fragments:
            fragment_results = self.db.search(fragment, top_k=1)
            if not fragment_results:
                continue
            _, entry = fragment_results[0]
            if entry["topic"] in seen_topics:
                continue
            seen_topics.add(entry["topic"])
            matched_entries.append(entry)

        if len(matched_entries) < 2:
            return None

        return self._format_multi_topic_reply(matched_entries)

    @staticmethod
    def _format_multi_topic_reply(entries: List[dict]) -> str:
        topic_names = [e["topic"] for e in entries]

        if len(topic_names) == 2:
            topics_joined = f"{topic_names[0]} and {topic_names[1]}"
            intro_phrase = "Here's both of them:"
        else:
            topics_joined = ", ".join(topic_names[:-1]) + f", and {topic_names[-1]}"
            intro_phrase = "Here's a quick look at each:"

        lines = [f"It seems like you're asking about {topics_joined}. {intro_phrase}"]
        for entry in entries:
            snapshot = AdvancedAI._first_sentence(entry["content"])
            lines.append(f"- **{entry['topic']}**: {snapshot}")
        lines.append("Ask one topic at once for a more detailed answer.")
        return "\n".join(lines)

    @staticmethod
    def _first_sentence(text: str) -> str:
        """Returns just the first sentence of a longer fact, for a brief
        snapshot rather than the full explanation."""
        match = re.match(r"(.+?[.!?])(\s|$)", text)
        return match.group(1) if match else text

    # ---------- fallback ----------

    _QUESTION_STARTERS = re.compile(
        r"^(who|what|when|where|why|how|which|do|does|did|is|are|can|could|"
        r"should|would|will|explain|define|describe)\b",
        re.IGNORECASE,
    )

    def _fallback(self, text: str = "") -> str:
        # A short remark that doesn't look like an actual question (no "?"
        # and doesn't start with a question word) is almost certainly just
        # casual chit-chat ("cool story", "haha nice", "aww") rather than
        # something the person expects a factual answer to — so give a
        # warm, natural reply instead of "that's not in my knowledge base".
        word_count = len(text.split())
        looks_like_question = "?" in text or bool(self._QUESTION_STARTERS.match(text.strip()))
        if text and word_count <= 5 and not looks_like_question:
            return self.small_talk.generic_reply()

        options = [
            "I don't have that in my knowledge base yet. You can teach me "
            "with 'remember X is Y', ask me a math question, or try "
            "rephrasing!",
            "Hmm, that one's not in my knowledge base. Try rewording it, "
            "ask me some math, or teach me with 'remember X is Y'.",
            "I'm not sure about that yet — want to teach me? Just say "
            "'remember <topic> is <fact>' and I'll remember it next time.",
        ]
        candidates = [o for o in options if o != self._last_fallback] or options
        choice = random.choice(candidates)
        self._last_fallback = choice
        return choice

    # ---------- utility ----------

    def stats(self) -> str:
        count = self.db.count()
        breakdown = ", ".join(f"{cat}: {n}" for cat, n in self.db.category_breakdown().items())
        return f"Knowledge base has {count} entries. Breakdown -> {breakdown}"

    def delete_last(self) -> str:
        """
        Removes the most recent (your message, my reply) pair from what I
        remember. Note: Python running in a terminal can't reach back and
        erase text that's already been printed to the screen — that's a
        limitation of how terminals work, not something any script can get
        around. This clears it from memory going forward instead, so it
        won't factor into anything from here on.
        """
        if not self.history:
            return "There's nothing to delete yet — we haven't exchanged anything."
        removed_input, removed_reply = self.history.pop()
        preview = removed_input if len(removed_input) <= 60 else removed_input[:57] + "..."
        return (
            f"Deleted our last exchange (\"{preview}\") from memory. I can't "
            "erase text already printed above, but it's forgotten from here on."
        )


# ============================================================
# 5. PERSISTENT QUERY COUNTER
# ============================================================
# Keeps a running total of how many questions ZAI has ever answered,
# saved to a small text file so the count survives between runs.
#
# The file lives next to this script (using its absolute path) rather
# than a bare relative filename — some online compilers change the
# working directory between runs, which would otherwise make a
# relative path point at nothing and silently reset the count.
#
# Important limitation: some online compilers/sandboxes (this can
# include certain Programiz configurations) wipe the entire filesystem
# between separate "Run" clicks, even though writes succeed *during*
# a run. If that's the case here, the counter will correctly go up
# while you're chatting, but will always start back at 0 the next time
# you run the script — that's a property of the environment, not a
# bug in this code, and there's no way for a plain Python script to
# work around it.

try:
    _SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _SCRIPT_DIR = os.getcwd()

STATS_FILE = os.path.join(_SCRIPT_DIR, "zai_stats.txt")


def _load_query_count() -> int:
    try:
        with open(STATS_FILE, "r") as f:
            return int(f.read().strip())
    except Exception:
        return 0


def _save_query_count(count: int) -> None:
    try:
        with open(STATS_FILE, "w") as f:
            f.write(str(count))
            f.flush()
    except Exception:
        pass  # read-only environment — just keep going without persistence


def _reset_query_count() -> None:
    """Wipes the saved question count back to 0, e.g. after testing or if
    a leftover stats file carried an old count into a fresh copy."""
    _save_query_count(0)


# ============================================================
# 6. COMMAND REGISTRY (for the 'cmds' command)
# ============================================================
# Single source of truth for every command's short description (shown in
# the 'cmds' list) and detailed explanation (shown by 'cmds <command>').

COMMANDS_INFO: "dict[str, dict[str, str]]" = {
    "stats": {
        "short": "Shows knowledge base entry counts",
        "usage": "stats",
        "detail": "Prints how many facts are stored in the knowledge base "
                  "in total, plus a breakdown by category (science, math, "
                  "english, general). Useful for seeing how much ZAI "
                  "currently knows.",
    },
    "more": {
        "short": "Shows version and creator info",
        "usage": "more",
        "detail": "Displays ZAI's version number, who made it (ZXStudios), "
                  "and which Python version it needs to run. Shown as a "
                  "plain block, without the usual 'ZAI:' prefix.",
    },
    "forget": {
        "short": "Forgets a fact you taught",
        "usage": "forget <topic>",
        "detail": "Removes a fact you previously taught ZAI with "
                  "'remember <topic> is <fact>'. It only works on things "
                  "you've taught during this session — ZAI's built-in "
                  "knowledge base can never be forgotten this way. "
                  "Example: 'forget octopi'.",
    },
    "delete": {
        "short": "Removes the last chat exchange",
        "usage": "delete",
        "detail": "Deletes your last message and ZAI's reply to it from "
                  "ZAI's memory, so it won't be treated as part of the "
                  "conversation going forward. Note: this can't erase text "
                  "already printed to the screen — that's a limitation of "
                  "how terminals work, not something a Python script can "
                  "control.",
    },
    "remember": {
        "short": "Teaches ZAI a new fact",
        "usage": "remember <topic> is <fact>",
        "detail": "Adds a brand-new fact to ZAI's knowledge base for this "
                  "session. Example: 'remember Singapore is a city-state "
                  "in Southeast Asia'. You can undo this later with the "
                  "'forget' command.",
    },
    "exit": {
        "short": "Ends the conversation",
        "usage": "exit",
        "detail": "Ends the chat and closes the program. 'quit' and 'bye' "
                  "work the same way.",
    },
    "cmds": {
        "short": "Lists all available commands",
        "usage": "cmds",
        "detail": "Shows a short description of every command. Add a "
                  "command name afterward (e.g. 'cmds forget') to see a "
                  "more detailed explanation of just that one.",
    },
    "resetstats": {
        "short": "Resets the question counter to 0",
        "usage": "resetstats",
        "detail": "Wipes the saved question count (the number shown in "
                  "the startup banner) back to 0. Useful after testing, or "
                  "if a leftover stats file from a previous copy carried "
                  "an old count into a fresh one.",
    },
}


# ============================================================
# 7. COMMAND-LINE CHAT LOOP
# ============================================================

def main() -> None:
    ai = AdvancedAI()
    query_count = _load_query_count()

    print("=" * 60)
    print(f"Made by ZXStudios | V1.2 | {query_count} of your questions were none too big for me")
    print("=" * 60)
    print("Type a question (science / math / English / general), a math")
    print("expression like '12 * (3 + 4)', or just chat.")
    print("Type 'cmds' to see all commands.")
    print("-" * 60)

    try:
        name = input("What's your name? ").strip()
    except (EOFError, KeyboardInterrupt):
        name = ""
    if not name:
        name = "Friend"

    prompt_label = f"{name} (You): "

    print(f"\nZAI: Nice to meet you, {name}! Ask me anything.")

    while True:
        try:
            user_input = input(f"\n{prompt_label}").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nZAI: Goodbye!")
            _save_query_count(query_count)
            break

        if not user_input:
            continue
        if user_input.lower() in {"exit", "quit", "bye"}:
            print("ZAI: Goodbye!")
            _save_query_count(query_count)
            break
        if user_input.lower() == "stats":
            print(f"ZAI: {ai.stats()}")
            continue
        if user_input.lower() == "more":
            # Shown with no "ZAI:" prefix, as its own standalone block.
            print(
                "Version Name: ZAI 1.2\n"
                "Made by ZXStudios\n"
                "Runs on Python 3.8 and above"
            )
            continue
        if user_input.lower() == "delete":
            print(f"ZAI: {ai.delete_last()}")
            continue
        if user_input.lower() == "resetstats":
            _reset_query_count()
            query_count = 0
            print("ZAI: Done — the question counter is back to 0.")
            continue
        if user_input.lower() == "cmds":
            longest_name = max(len(name) for name in COMMANDS_INFO)
            lines = ["ZAI: Here are my commands:"]
            for name, info in COMMANDS_INFO.items():
                lines.append(f"  {name.ljust(longest_name)} - {info['short']}")
            print("\n".join(lines))
            print('\nAsk me "cmds <command>" for more info.')
            continue
        if user_input.lower().startswith("cmds "):
            target = user_input[len("cmds "):].strip().lower()
            info = COMMANDS_INFO.get(target)
            if info:
                print(f"ZAI: {info['usage']}\n{info['detail']}")
            else:
                print(
                    f"ZAI: I don't have a command called '{target}'. "
                    "Type 'cmds' to see the full list."
                )
            continue

        reply = ai.process(user_input)
        print(f"ZAI: {reply}")

        query_count += 1
        _save_query_count(query_count)


if __name__ == "__main__":
    main()

Made by ZXStudios | V1.2 | 11 of your questions were none too big for me
Type a question (science / math / English / general), a math
expression like '12 * (3 + 4)', or just chat.
Type 'cmds' to see all commands.
------------------------------------------------------------
